# 🤖 ActuarIA — Agent A7 : Provisionnement v3

**Nouveautés v3 :**
- Validation des hypothèses avant calcul (H1 indépendance · H2 stabilité · H3 a priori BF)
- Score de confiance par méthode (0-100%)
- Recommandation automatique de méthode
- 4 graphiques Plotly : heatmap triangle · facteurs CL ±2σ · IBNR · convergence

**Compatibilité :** Données client (result_a2) ou triangle externe (numpy array)

---

In [1]:
# CELLULE 1 — MONTAGE DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive monté')

Mounted at /content/drive
✅ Drive monté


In [2]:
# CELLULE 2 — INSTALLATION + CHARGEMENT AGENTS
!pip install plotly -q

%run '/content/drive/MyDrive/ActuarIA/agents/a1_ingestion.py'
%run '/content/drive/MyDrive/ActuarIA/agents/a2_preprocessing.py'
%run '/content/drive/MyDrive/ActuarIA/agents/a7_provisionnement.py'
print('✅ Agents A1, A2, A7 v3 chargés')

Agent A1 — Ingestion ActuarIA v2.0
Nouveautés : config centrale · mapping client · portabilité

Usage standard :
  agent_a1 = AgentA1Ingestion()
  result_a1 = agent_a1.run(branche='non_vie', fichier='contrats.parquet')

Usage avec client réel :
  creer_mapping_client('client_xyz', {'NUM_POL': 'id_contrat', ...})
  result_a1 = agent_a1.run(branche='non_vie', fichier='data.xlsx', client_id='client_xyz')
Agent A2 — Preprocessing & Feature Engineering ActuarIA v1.0
Importez ce module dans votre notebook Colab.
Exemple : %run 'chemin/a2_preprocessing.py'
Agent A7 — Provisionnement ActuarIA v2.0
Nouveautés : valeurs extrêmes · BF personnalisable · méthodes facteurs

Usage standard (v1 compatible) :
  result_a7 = agent_a7.run(result_a2=result_a2)

Usage avancé v2 :
  result_a7 = agent_a7.run(
      result_a2        = result_a2,
      methode_facteurs = 'volume_weighted',
      taux_bf_manuel   = 0.75,
      annees_a_exclure = [0],
      seuil_alerte     = 2.0,
  )
✅ Agents A1, A2, A7 v3 charg

In [3]:
# CELLULE 3 — CONFIGURATION
BASE_PATH   = '/content/drive/MyDrive/ActuarIA'
DATA_PATH   = f'{BASE_PATH}/data'
MODELS_PATH = f'{BASE_PATH}/models'
AUDIT_PATH  = f'{BASE_PATH}/audit'
BRANCHE     = 'non_vie'
FICHIER     = 'contrats_auto_70k.parquet'

# ── PARAMÈTRES A7 v3 ─────────────────────────────────────────────────────────
METHODE_FACTEURS    = 'standard'   # standard · volume_weighted · mediane · trimmed_mean
TAUX_BF_MANUEL      = None         # None = auto | 0.75 = 75% forcé
ANNEES_A_EXCLURE    = None         # None = auto | [2] = exclure année index 2
SEUIL_ALERTE        = 2.0          # Seuil détection atypiques (σ)
VALIDER_HYPOTHESES  = True         # Valider H1/H2/H3 avant calcul
GENERER_GRAPHIQUES  = True         # Générer les 4 graphiques Plotly

print(f'✅ Config OK | Méthode : {METHODE_FACTEURS} | Validation : {VALIDER_HYPOTHESES} | Graphiques : {GENERER_GRAPHIQUES}')

✅ Config OK | Méthode : standard | Validation : True | Graphiques : True


In [4]:
# CELLULE 4 — PIPELINE A1 → A2 → A7 v3
print('A1...')
agent_a1  = AgentA1Ingestion(base_path=DATA_PATH, audit_path=AUDIT_PATH, verbose=False)
result_a1 = agent_a1.run(branche=BRANCHE, fichier=FICHIER)
print(f'✅ A1 : {result_a1["statut_rag"]} | {len(result_a1["dataframe"]):,} lignes')

print('A2...')
agent_a2  = AgentA2Preprocessing(models_path=MODELS_PATH, audit_path=AUDIT_PATH, verbose=False)
result_a2 = agent_a2.run(result_a1)
print(f'✅ A2 : {result_a2["statut_rag"]}')

print('A7 v3 — Provisionnement + Validation + Graphiques...')
agent_a7  = AgentA7Provisionnement(models_path=MODELS_PATH, audit_path=AUDIT_PATH, verbose=True)
result_a7 = agent_a7.run(
    result_a2           = result_a2,
    methode_facteurs    = METHODE_FACTEURS,
    taux_bf_manuel      = TAUX_BF_MANUEL,
    annees_a_exclure    = ANNEES_A_EXCLURE,
    seuil_alerte        = SEUIL_ALERTE,
    valider_hypotheses  = VALIDER_HYPOTHESES,
    generer_graphiques  = GENERER_GRAPHIQUES,
)

A1...
✅ A1 : VERT | 70,000 lignes
A2...
✅ A2 : AMBRE
A7 v3 — Provisionnement + Validation + Graphiques...

═════════════════════════════════════════════════════════════════
  ACTUARIA — AGENT A7 PROVISIONNEMENT | A7_20260621_133823
═════════════════════════════════════════════════════════════════
  🟢 STATUT : VERT

═════════════════════════════════════════════════════════════════
  🟢 PROVISIONNEMENT v2 — VERT
  Sous-branche : auto
  Méthode facteurs : standard
  Taux BF : auto
  
  BEST ESTIMATE S2 : 2,921,176 €
  CV inter-méthodes : 0.3%
  Provision P90 : 2,921,176 €
  
  DIAGNOSTIC :
  Triangle sans anomalie détectée. Les 4 méthodes convergent avec un CV de 0.3%.
  
  RECOMMANDATION :
  → Best Estimate retenu pour le bilan S2.
  → Documenter la méthode de facteurs dans l'audit trail.
═════════════════════════════════════════════════════════════════



In [5]:
# CELLULE 5 — VALIDATION DES HYPOTHÈSES
v = result_a7['validation']

print('VALIDATION DES HYPOTHÈSES ACTUARIELLES')
print('=' * 55)
print()

# H1 — Indépendance
h1 = v['h1_independance']
statut_h1 = '✅ VALIDÉE' if h1['ok'] else '❌ VIOLÉE'
print(f"H1 — Indépendance des années : {statut_h1}")
print(f"     Corrélation max = {h1['corr_max']:.3f} (seuil 0.70) | Score = {h1['score']}/100")
print()

# H2 — Stabilité
h2 = v['h2_stabilite']
statut_h2 = '✅ VALIDÉE' if h2['ok'] else '⚠️  MARGINALE'
print(f"H2 — Stabilité des facteurs  : {statut_h2}")
print(f"     CV moyen = {h2['cv_moy']:.3f} (seuil 0.30) | Score = {h2['score']}/100")
print()

# H3 — A priori BF
h3 = v['h3_apriori_bf']
print(f"H3 — Qualité a priori BF     : Score = {h3['score']}/100")
print()

# Scores et recommandation
print('SCORES DE CONFIANCE PAR MÉTHODE')
print('-' * 40)
for methode, score in sorted(v['scores_confiance'].items(), key=lambda x: -x[1]):
    barre = '█' * (score // 10) + '░' * (10 - score // 10)
    tag   = ' ← RECOMMANDÉE ⭐' if methode == v['methode_recommandee'] else ''
    print(f"  {methode:<15} : [{barre}] {score}/100{tag}")
print()
print(f"⭐ RECOMMANDATION : {v['methode_recommandee'].upper()}")
print(f"   {v['recommandation']}")
print()

# Alertes
if v['alertes']:
    print('ALERTES :')
    for a in v['alertes']:
        print(f'  {a}')
else:
    print('✅ Aucune alerte — toutes les hypothèses validées')

VALIDATION DES HYPOTHÈSES ACTUARIELLES

H1 — Indépendance des années : ❌ VIOLÉE
     Corrélation max = 1.000 (seuil 0.70) | Score = 0/100

H2 — Stabilité des facteurs  : ✅ VALIDÉE
     CV moyen = 0.000 (seuil 0.30) | Score = 99/100

H3 — Qualité a priori BF     : Score = 99/100

SCORES DE CONFIANCE PAR MÉTHODE
----------------------------------------
  bf              : [██████░░░░] 69/100 ← RECOMMANDÉE ⭐
  cape_cod        : [█████░░░░░] 50/100
  chain_ladder    : [████░░░░░░] 49/100
  mack_1993       : [████░░░░░░] 49/100

⭐ RECOMMANDATION : BF
   Bornhuetter-Ferguson — triangle court ou données limitées

ALERTES :
  ⚠️  H1 Indépendance : corrélation max = 1.00 > 0.70 → Années de survenance potentiellement dépendantes


In [6]:
# CELLULE 6 — GRAPHIQUE 1 : TRIANGLE HEATMAP
if result_a7['graphiques'].get('heatmap_triangle'):
    print('📊 Triangle de développement — Heatmap')
    result_a7['graphiques']['heatmap_triangle'].show()
else:
    print('Graphique non disponible — vérifier que plotly est installé')

📊 Triangle de développement — Heatmap


In [7]:
# CELLULE 7 — GRAPHIQUE 2 : FACTEURS CL AVEC BANDES ±2σ
if result_a7['graphiques'].get('facteurs_cl'):
    print('📊 Facteurs de développement avec bandes ±2σ')
    result_a7['graphiques']['facteurs_cl'].show()
else:
    print('Graphique non disponible')

📊 Facteurs de développement avec bandes ±2σ


In [8]:
# CELLULE 8 — GRAPHIQUE 3 : IBNR PAR ANNÉE
if result_a7['graphiques'].get('ibnr_par_annee'):
    print('📊 IBNR par année de survenance')
    result_a7['graphiques']['ibnr_par_annee'].show()
else:
    print('Graphique non disponible')

📊 IBNR par année de survenance


In [9]:
# CELLULE 9 — GRAPHIQUE 4 : CONVERGENCE DES 4 MÉTHODES
if result_a7['graphiques'].get('convergence_methodes'):
    print('📊 Convergence des 4 méthodes avec IC Mack 95%')
    result_a7['graphiques']['convergence_methodes'].show()
else:
    print('Graphique non disponible')

📊 Convergence des 4 méthodes avec IC Mack 95%


In [10]:
# CELLULE 10 — RÉSULTATS COMPLETS
import pandas as pd
be = result_a7['best_estimate']

print('RÉSULTATS PROVISIONNEMENT A7 v3')
print('=' * 55)
rows = [
    {'Méthode':'Chain Ladder',         'Réserve (€)':f"{result_a7['chain_ladder']['reserve_totale']:,.0f}",  'Confiance':f"{result_a7['validation']['scores_confiance']['chain_ladder']}/100"},
    {'Méthode':'Mack 1993',            'Réserve (€)':f"{result_a7['mack']['reserve_best_estimate']:,.0f}",   'Confiance':f"{result_a7['validation']['scores_confiance']['mack_1993']}/100"},
    {'Méthode':'Bornhuetter-Ferguson', 'Réserve (€)':f"{result_a7['bf']['reserve_totale']:,.0f}",            'Confiance':f"{result_a7['validation']['scores_confiance']['bf']}/100"},
    {'Méthode':'Cape Cod',             'Réserve (€)':f"{result_a7['cape_cod']['reserve_totale']:,.0f}",       'Confiance':f"{result_a7['validation']['scores_confiance']['cape_cod']}/100"},
    {'Méthode':'── BEST ESTIMATE S2 ──','Réserve (€)':f"{be['best_estimate']:,.0f}",                          'Confiance':'100%'},
]
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"CV inter-méthodes : {be['cv_inter_methodes']:.2f}%")
print(f"Provision P90     : {be['reserve_p90']:,.0f} €")
print(f"Statut RAG        : {result_a7['statut_rag']}")

RÉSULTATS PROVISIONNEMENT A7 v3
               Méthode Réserve (€) Confiance
          Chain Ladder   2,928,003    49/100
             Mack 1993   2,928,003    49/100
  Bornhuetter-Ferguson   2,908,802    69/100
              Cape Cod   2,917,338    50/100
── BEST ESTIMATE S2 ──   2,921,176      100%

CV inter-méthodes : 0.28%
Provision P90     : 2,921,176 €
Statut RAG        : VERT


In [11]:
# CELLULE 11 — TEST TRIANGLE CLIENT (optionnel)
# Remplace C_client par le vrai triangle du client
import numpy as np

# Exemple avec triangle client fourni directement
C_client = np.array([
    [1000, 1450, 1680, 1750],
    [1100, 1580, 1820,    0],
    [1050, 8500,    0,    0],  # Sinistre catastrophique
    [ 980,    0,    0,    0],
], dtype=float)

r_client = agent_a7.run(
    triangle            = C_client,
    methode_facteurs    = 'mediane',   # Robuste sur catastrophes
    valider_hypotheses  = True,
    generer_graphiques  = True,
)

print(f"Statut : {r_client['statut_rag']}")
print(f"BE     : {r_client['best_estimate']['best_estimate']:,.0f} €")
print(f"Recommandation : {r_client['validation']['methode_recommandee']}")
r_client['graphiques']['heatmap_triangle'].show()
r_client['graphiques']['facteurs_cl'].show()


═════════════════════════════════════════════════════════════════
  ACTUARIA — AGENT A7 PROVISIONNEMENT | A7_20260621_133826
═════════════════════════════════════════════════════════════════
  🟡 STATUT : AMBRE

═════════════════════════════════════════════════════════════════
  🟡 PROVISIONNEMENT v2 — AMBRE
  Sous-branche : auto
  Méthode facteurs : mediane
  Taux BF : auto
  
  BEST ESTIMATE S2 : 2,509 €
  CV inter-méthodes : 1.1%
  Provision P90 : 2,528 €
  
  ⚠️  ANNÉES ATYPIQUES EXCLUES : [2]
  
  ALERTES FACTEURS :
    🔴 ROUGE Facteur atypique : année 2, colonne 0 — f=8.095 (médiane=1.450) → ratio max/médiane = 5.6x (seuil=3.0x)
  
  DIAGNOSTIC :
  1 année(s) atypique(s) détectée(s). 
  ✅ CORRECTION APPLIQUÉE : méthode 'mediane' utilisée à la place de 'standard'.
     BE robuste (mediane) = 2,509 €
     BE standard (non corrigé) = potentiellement surestimé.
  Les provisions ont été recalculées avec la méthode robuste. L'actuaire doit valider les exclusions avant signature.
  
  RE

## Nouvelles méthodes avancées — v3

### Bootstrap Stochastique (England & Verrall 2002)
### Munich Chain Ladder (Quarg & Mack 2004)
### Comparaison N vs N-1


In [12]:
# Bootstrap Stochastique — Distribution des réserves
boot = result_a7['bootstrap']
print('BOOTSTRAP STOCHASTIQUE — 1 000 simulations')
print('=' * 50)
print(f"Méthode         : {boot['methode']}")
print(f"BE bootstrap    : {boot['be_bootstrap']:>12,.0f} euros")
print(f"Ecart-type      : {boot['std_bootstrap']:>12,.0f} euros")
print(f"CV bootstrap    : {boot['cv_bootstrap']*100:>11.1f}%")
print()
print('Distribution des réserves :')
print(f"  P50  (médiane) : {boot['p50']:>12,.0f} euros")
print(f"  P75            : {boot['p75']:>12,.0f} euros")
print(f"  P90            : {boot['p90']:>12,.0f} euros")
print(f"  P95            : {boot['p95']:>12,.0f} euros")
print(f"  P99.5 (VaR S2) : {boot['p99_5']:>12,.0f} euros")
print()
print(f"IC 95% : [{boot['ic_95_inf']:,.0f} euros — {boot['ic_95_sup']:,.0f} euros]")


BOOTSTRAP STOCHASTIQUE — 1 000 simulations
Méthode         : Bootstrap ODP — England & Verrall (2002)
BE bootstrap    :    2,928,003 euros
Ecart-type      :            0 euros
CV bootstrap    :         0.0%

Distribution des réserves :
  P50  (médiane) :    2,928,003 euros
  P75            :    2,928,003 euros
  P90            :    2,928,003 euros
  P95            :    2,928,003 euros
  P99.5 (VaR S2) :    2,928,003 euros

IC 95% : [2,928,003 euros — 2,928,003 euros]


In [13]:
# Graphique distribution bootstrap
import plotly.graph_objects as go
import numpy as np

dist = boot['distribution']
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=dist, nbinsx=50,
    marker_color='rgba(201,168,76,0.7)',
    marker_line=dict(color='#0F2E52', width=0.5),
    name='Simulations',
))
# Lignes percentiles
for p, label, color in [
    (boot['p50'],   'P50',    '#2ECC71'),
    (boot['p90'],   'P90',    '#F39C12'),
    (boot['p99_5'], 'P99.5',  '#E74C3C'),
]:
    fig.add_vline(x=p, line_color=color, line_width=2, line_dash='dash',
                  annotation_text=f'{label} = {p:,.0f}€',
                  annotation_font=dict(color=color, size=10))
fig.update_layout(
    paper_bgcolor='#0F2E52', plot_bgcolor='#1B3A5C',
    font=dict(family='Inter', color='#F0F4F8', size=11),
    title=dict(text='Distribution Bootstrap des réserves IBNR (1 000 simulations)',
               font=dict(color='#F0F4F8', size=13), x=0.01),
    xaxis=dict(title='Réserve IBNR (€)', tickfont=dict(color='#8A9AB0'),
               showgrid=True, gridcolor='rgba(255,255,255,0.05)'),
    yaxis=dict(title='Fréquence', tickfont=dict(color='#8A9AB0'),
               showgrid=True, gridcolor='rgba(255,255,255,0.05)'),
    showlegend=False, height=350,
)
fig.show()


In [14]:
# Munich Chain Ladder
# Note : nécessite deux triangles (fréquence ET coût)
# Ici on démontre avec des triangles synthétiques
import numpy as np

# Triangles de démonstration fréquence / coût
n = 5
C_freq = np.array([
    [100, 150, 165, 170, 172],
    [110, 160, 178, 185, 0  ],
    [105, 158, 172, 0,   0  ],
    [ 98, 145, 0,   0,   0  ],
    [115, 0,   0,   0,   0  ],
], dtype=float)

C_cout = np.array([
    [500, 620, 680, 710, 720],
    [520, 640, 705, 730, 0  ],
    [510, 630, 695, 0,   0  ],
    [480, 600, 0,   0,   0  ],
    [540, 0,   0,   0,   0  ],
], dtype=float)

munich = agent_a7._munich_chain_ladder(C_freq, C_cout)
print('MUNICH CHAIN LADDER (Quarg & Mack 2004)')
print('=' * 50)
print(f"Réserve Munich         : {munich['reserve_munich']:>10,.0f}")
print(f"Réserve Munich fréq.   : {munich['reserve_munich_freq']:>10,.0f}")
print(f"Réserve Munich coût    : {munich['reserve_munich_cout']:>10,.0f}")
print(f"Réserve CL fréq.       : {munich['reserve_cl_freq']:>10,.0f}")
print(f"Réserve CL coût        : {munich['reserve_cl_cout']:>10,.0f}")
print(f"Ecart Munich vs CL     : {munich['ecart_munich_cl']:>+10,.0f}")
print(f"Méthode : {munich['methode']}")


MUNICH CHAIN LADDER (Quarg & Mack 2004)
Réserve Munich         :        271
Réserve Munich fréq.   :         99
Réserve Munich coût    :        444
Réserve CL fréq.       :        114
Réserve CL coût        :        380
Ecart Munich vs CL     :        +24
Méthode : Munich Chain Ladder — Quarg & Mack (2004)


In [15]:
# Comparaison N vs N-1
n1 = result_a7['comparaison_n1']
print('COMPARAISON N vs N-1')
print('=' * 50)
print(f"BE N           : {n1['be_n']:>12,.0f} euros")
print(f"BE N-1         : {n1['be_n1']:>12,.0f} euros")
print(f"Variation abs. : {n1['variation_abs']:>+12,.0f} euros")
print(f"Variation %    : {n1['variation_pct']:>+11.1f}%")
print(f"Statut         : {n1['statut_evolution']}")
print()
print('Décomposition Waterfall :')
w = n1['waterfall']
print(f"  BE N-1                  : {w['be_n1']:>12,.0f} euros")
print(f"  + Effet run-off         : {w['effet_run_off']:>+12,.0f} euros")
print(f"  + Nouveaux sinistres    : {w['effet_nouveaux']:>+12,.0f} euros")
print(f"  + Réouvertures          : {w['effet_reouverture']:>+12,.0f} euros")
print(f"  + Changement hypothèses : {w['effet_hypotheses']:>+12,.0f} euros")
print(f"  + Effet résiduel        : {w['effet_residuel']:>+12,.0f} euros")
print(f"  = BE N                  : {w['be_n']:>12,.0f} euros")
print()
print(f"Interprétation : {n1['interpretation']}")


COMPARAISON N vs N-1
BE N           :    2,921,176 euros
BE N-1         :    2,833,541 euros
Variation abs. :      +87,635 euros
Variation %    :        +3.1%
Statut         : VERT

Décomposition Waterfall :
  BE N-1                  :    2,833,541 euros
  + Effet run-off         :     -425,031 euros
  + Nouveaux sinistres    :     +292,118 euros
  + Réouvertures          :      +17,527 euros
  + Changement hypothèses :      +26,291 euros
  + Effet résiduel        :     +176,731 euros
  = BE N                  :    2,921,176 euros

Interprétation : Les provisions ont augmenté de 3.1% entre N-1 et N. Statut : VERT.


In [16]:
# Graphique Waterfall N vs N-1
w = result_a7['comparaison_n1']['waterfall']
labels = ['BE N-1', 'Run-off', 'Nouveaux', 'Réouvertures', 'Hypothèses', 'Résiduel', 'BE N']
values = [w['be_n1'], w['effet_run_off'], w['effet_nouveaux'],
          w['effet_reouverture'], w['effet_hypotheses'], w['effet_residuel'], w['be_n']]
colors = ['#C9A84C' if i in [0, 6] else '#2ECC71' if v >= 0 else '#E74C3C'
          for i, v in enumerate(values)]

fig = go.Figure(go.Bar(
    x=labels, y=values,
    marker_color=colors,
    marker_line=dict(color='#0F2E52', width=1),
    width=0.45, opacity=0.88,
    text=[f"{v:+,.0f}€" if i not in [0,6] else f"{v:,.0f}€" for i,v in enumerate(values)],
    textposition='outside',
    textfont=dict(color='#F0F4F8', size=10),
))
fig.update_layout(
    paper_bgcolor='#0F2E52', plot_bgcolor='#1B3A5C',
    font=dict(family='Inter', color='#F0F4F8', size=11),
    title=dict(text='Waterfall Provisions N-1 → N (décomposition des écarts)',
               font=dict(color='#F0F4F8', size=13), x=0.01),
    xaxis=dict(tickfont=dict(color='#F0F4F8', size=10), showgrid=False),
    yaxis=dict(visible=False), bargap=0.35, showlegend=False, height=320,
)
fig.show()
